# Exploration notebook - CS611 Assignment 1

- scratch space used to work out the bronze/silver/gold pipeline before it was turned into `.py` scripts
- final production logic lives in `utils/` and is run via `main.py`
- this notebook re-uses those same functions (imported, not copy-pasted) so it can't drift out of sync


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("explore").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")


## 1. Load raw data, check schema

- 4 raw csv files: clickstream, loan_daily, financials, attributes
- just eyeballing schema + a few rows first


In [2]:
df = spark.read.csv("data/feature_clickstream.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)


root
 |-- fe_1: integer (nullable = true)
 |-- fe_2: integer (nullable = true)
 |-- fe_3: integer (nullable = true)
 |-- fe_4: integer (nullable = true)
 |-- fe_5: integer (nullable = true)
 |-- fe_6: integer (nullable = true)
 |-- fe_7: integer (nullable = true)
 |-- fe_8: integer (nullable = true)
 |-- fe_9: integer (nullable = true)
 |-- fe_10: integer (nullable = true)
 |-- fe_11: integer (nullable = true)
 |-- fe_12: integer (nullable = true)
 |-- fe_13: integer (nullable = true)
 |-- fe_14: integer (nullable = true)
 |-- fe_15: integer (nullable = true)
 |-- fe_16: integer (nullable = true)
 |-- fe_17: integer (nullable = true)
 |-- fe_18: integer (nullable = true)
 |-- fe_19: integer (nullable = true)
 |-- fe_20: integer (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe

In [3]:
df2 = spark.read.csv("data/lms_loan_daily.csv", header=True, inferSchema=True)
df2.printSchema()
df2.show(5)


root
 |-- loan_id: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- loan_start_date: date (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- installment_num: integer (nullable = true)
 |-- loan_amt: integer (nullable = true)
 |-- due_amt: double (nullable = true)
 |-- paid_amt: double (nullable = true)
 |-- overdue_amt: double (nullable = true)
 |-- balance: double (nullable = true)
 |-- snapshot_date: date (nullable = true)

+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|             loan_id|Customer_ID|loan_start_date|tenure|installment_num|loan_amt|due_amt|paid_amt|overdue_amt|balance|snapshot_date|
+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|CUS_0x1000_2023_0...| CUS_0x1000|     2023-05-01|    10|              0|   10000|    0.0|     0.0|        0.0|10000.0|   2023-05-01|

In [4]:
df3 = spark.read.csv("data/features_financials.csv", header=True, inferSchema=True)
df3.printSchema()
df3.show(5)


root
 |-- Customer_ID: string (nullable = true)
 |-- Annual_Income: string (nullable = true)
 |-- Monthly_Inhand_Salary: double (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: string (nullable = true)
 |-- Type_of_Loan: string (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: string (nullable = true)
 |-- Changed_Credit_Limit: string (nullable = true)
 |-- Num_Credit_Inquiries: double (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: string (nullable = true)
 |-- Credit_Utilization_Ratio: double (nullable = true)
 |-- Credit_History_Age: string (nullable = true)
 |-- Payment_of_Min_Amount: string (nullable = true)
 |-- Total_EMI_per_month: double (nullable = true)
 |-- Amount_invested_monthly: string (nullable = true)
 |-- Payment_Behaviour: string (nullable = true)
 |-- Mont

In [5]:
df4 = spark.read.csv("data/features_attributes.csv", header=True, inferSchema=True)
df4.printSchema()
df4.show(5)


root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

+-----------+--------------+---+-----------+-------------+-------------+
|Customer_ID|          Name|Age|        SSN|   Occupation|snapshot_date|
+-----------+--------------+---+-----------+-------------+-------------+
| CUS_0x1000|Alistair Barrf| 18|913-74-1218|       Lawyer|   2023-05-01|
| CUS_0x1009|        Arunah| 26|063-67-6938|     Mechanic|   2025-01-01|
| CUS_0x100b|      Shirboni| 19|  #F%$D@*&8|Media_Manager|   2024-03-01|
| CUS_0x1011|     Schneyerh| 44|793-05-8223|       Doctor|   2023-11-01|
| CUS_0x1013|      Cameront| 44|930-49-9615|     Mechanic|   2023-12-01|
+-----------+--------------+---+-----------+-------------+-------------+
only showing top 5 rows



## 2. Row counts & snapshot coverage

- checking how many monthly snapshots each table has, and the date range
- clickstream/loan_daily go further back/forward than financials/attributes - worth knowing before joining


In [6]:
for name, d in [("clickstream", df), ("loan_daily", df2), ("financials", df3), ("attributes", df4)]:
    dates = sorted(r["snapshot_date"] for r in d.select("snapshot_date").distinct().collect())
    print(f"{name:12s} rows={d.count():>7d}  snapshot_dates={len(dates):>3d}  range=({dates[0]} -> {dates[-1]})")


clickstream  rows= 215376  snapshot_dates= 24  range=(2023-01-01 -> 2024-12-01)
loan_daily   rows= 137500  snapshot_dates= 35  range=(2023-01-01 -> 2025-11-01)
financials   rows=  12500  snapshot_dates= 25  range=(2023-01-01 -> 2025-01-01)
attributes   rows=  12500  snapshot_dates= 25  range=(2023-01-01 -> 2025-01-01)


## 3. Distinct customer / loan counts

- sanity check: how many unique customers and loans are we actually dealing with


In [7]:
print("financials distinct customers:", df3.select("Customer_ID").distinct().count())
print("attributes distinct customers:", df4.select("Customer_ID").distinct().count())
print("clickstream distinct customers:", df.select("Customer_ID").distinct().count())
print("loan_daily distinct customers:", df2.select("Customer_ID").distinct().count())
print("loan_daily distinct loans:", df2.select("loan_id").distinct().count())


financials distinct customers: 12500
attributes distinct customers: 12500
clickstream distinct customers: 8974
loan_daily distinct customers: 12500
loan_daily distinct loans: 12500


## 4. Tenure check

- need to know if loan tenure is fixed or variable, to pick a sensible mob (months-on-book) checkpoint for the label
- turns out every loan has tenure = 10 months, fully observed, no censoring


In [8]:
tenure_dist = df2.select("tenure").distinct().orderBy("tenure").collect()
print([r["tenure"] for r in tenure_dist])

max_installment = df2.groupBy("loan_id").agg({"installment_num": "max"}).collect()
import statistics
vals = [r["max(installment_num)"] for r in max_installment]
print("min max_installment:", min(vals), " median:", statistics.median(vals), " max:", max(vals))


[10]
min max_installment: 10  median: 10.0  max: 10


## 5. Snapshot-date alignment check

- checking whether `loan_start_date` (loan_daily) lines up with `snapshot_date` in financials/clickstream
- this matters for the feature store join - want to join on loan origination date only, not any random snapshot, to avoid leakage


In [9]:
import pyspark.sql.functions as F

check = (
    df2.select("Customer_ID", "loan_start_date")
       .distinct()
       .join(df3.select("Customer_ID", "snapshot_date"), on="Customer_ID", how="inner")
       .withColumn("dates_match", F.col("loan_start_date") == F.col("snapshot_date"))
)
check.groupBy("dates_match").count().show()


+-----------+-----+
|dates_match|count|
+-----------+-----+
|       true|12500|
+-----------+-----+



In [10]:
check2 = (
    df2.select("Customer_ID", "loan_start_date")
       .distinct()
       .join(df.select("Customer_ID", "snapshot_date"), on="Customer_ID", how="inner")
       .withColumn("dates_match", F.col("loan_start_date") == F.col("snapshot_date"))
)
check2.groupBy("dates_match").count().show()


+-----------+------+
|dates_match| count|
+-----------+------+
|       true|  8974|
|      false|206402|
+-----------+------+



## 6. Raw value inspection

- eyeballing raw (pre-cleaning) values for the columns that turned out to have placeholder junk
- e.g. Credit_Mix has a stray "_", Payment_of_Min_Amount has "NM", Occupation has "_______"
- this is what drove the cleaning rules in `utils/data_processing_silver.py`


In [11]:
df3.select("Annual_Income").distinct().orderBy(F.rand()).show(10, truncate=False)
df3.select("Num_of_Loan").distinct().show(10, truncate=False)
df3.select("Credit_Mix").distinct().show()
df3.select("Payment_of_Min_Amount").distinct().show()
df4.select("Age").distinct().orderBy(F.rand()).show(10, truncate=False)
df4.select("Occupation").distinct().show(20, truncate=False)


+------------------+
|Annual_Income     |
+------------------+
|20168.99          |
|86638.52          |
|12763.795         |
|104118.69         |
|14156.88          |
|12629.6           |
|1034126.0         |
|34829.92_         |
|88027.62          |
|109744.86000000002|
+------------------+
only showing top 10 rows

+-----------+
|Num_of_Loan|
+-----------+
|1159       |
|7          |
|1_         |
|1171_      |
|1412       |
|387        |
|87         |
|3          |
|966        |
|365        |
+-----------+
only showing top 10 rows

+----------+
|Credit_Mix|
+----------+
|         _|
|      Good|
|       Bad|
|  Standard|
+----------+

+---------------------+
|Payment_of_Min_Amount|
+---------------------+
|                   NM|
|                   No|
|                  Yes|
+---------------------+

+-----+
|Age  |
+-----+
|18_  |
|18   |
|7723_|
|15   |
|8678 |
|2090 |
|52_  |
|5714 |
|2932 |
|19   |
+-----+
only showing top 10 rows

+-------------+
|Occupation   |
+-------------

## 7. Bronze + silver build

- from here on, use the actual pipeline functions from `utils/` instead of re-writing them here
- keeps this notebook honest - it can't quietly drift out of sync with the graded `.py` files


In [12]:
import os
from utils.data_processing_bronze import process_bronze_table, RAW_FILE_MAP
from utils.data_processing_silver import SILVER_PROCESSORS

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
DATAMART_DIR = os.path.join(BASE_DIR, "datamart")
BRONZE_DIR = os.path.join(DATAMART_DIR, "bronze")
SILVER_DIR = os.path.join(DATAMART_DIR, "silver")
GOLD_DIR = os.path.join(DATAMART_DIR, "gold")

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)
os.makedirs(GOLD_DIR, exist_ok=True)


def get_snapshot_dates(table_name, spark):
    raw_path = os.path.join(DATA_DIR, RAW_FILE_MAP[table_name])
    df = spark.read.csv(raw_path, header=True, inferSchema=True)
    dates = [r["snapshot_date"] for r in df.select("snapshot_date").distinct().collect()]
    return sorted(str(d) for d in dates)


for table_name in RAW_FILE_MAP:
    dates = get_snapshot_dates(table_name, spark)
    print(f"\n=== {table_name}: {len(dates)} snapshot dates ===")
    silver_fn = SILVER_PROCESSORS[table_name]
    for d in dates:
        bronze_path = process_bronze_table(table_name, d, DATA_DIR, BRONZE_DIR, spark)
        bronze_df = spark.read.csv(bronze_path, header=True, inferSchema=True)
        silver_fn(bronze_df, d, SILVER_DIR)



=== clickstream: 24 snapshot dates ===
[bronze] clickstream  2023-01-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_01_01.csv
[silver] clickstream  2023-01-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_01_01.parquet
[bronze] clickstream  2023-02-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_02_01.csv
[silver] clickstream  2023-02-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_02_01.parquet
[bronze] clickstream  2023-03-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_03_01.csv
[silver] clickstream  2023-03-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_03_01.parquet
[bronze] clickstream  2023-04-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_04_01.csv
[silver] clickstream  2023-04-01:   8974 rows

## 8. Gold layer: label store + feature store

- same idea - import the real functions from `utils/data_processing_gold.py`, don't re-define them here


In [13]:
from utils.data_processing_gold import process_gold_label_store, process_gold_feature_store

silver_loan_daily_all = spark.read.parquet(os.path.join(SILVER_DIR, "loan_daily", "*.parquet"))
silver_attributes_all = spark.read.parquet(os.path.join(SILVER_DIR, "attributes", "*.parquet"))
silver_financials_all = spark.read.parquet(os.path.join(SILVER_DIR, "financials", "*.parquet"))
silver_clickstream_all = spark.read.parquet(os.path.join(SILVER_DIR, "clickstream", "*.parquet"))

label_store = process_gold_label_store(silver_loan_daily_all, GOLD_DIR)
feature_store = process_gold_feature_store(silver_attributes_all, silver_financials_all, silver_clickstream_all, GOLD_DIR)


[gold] label_store:  12500 rows, bad rate=0.288 -> /home/jovyan/work/datamart/gold/label_store/gold_label_store.parquet
[gold] feature_store:  12500 rows -> /home/jovyan/work/datamart/gold/feature_store/gold_feature_store.parquet


## 9. Sanity check: label + feature join

- checking the two gold tables actually join cleanly on (Customer_ID, loan_start_date = snapshot_date)
- row count should match label_store row count exactly if the join is aligned


In [14]:
joined = label_store.join(
    feature_store,
    on=(label_store["Customer_ID"] == feature_store["Customer_ID"]) &
       (label_store["loan_start_date"] == feature_store["snapshot_date"]),
    how="inner",
)
print("joined rows:", joined.count())


joined rows: 12500


## 10. Sanity-check model

- quick and dirty logistic regression, just to confirm the features aren't completely uninformative
- not a real model, no tuning, no proper eval - just a smoke test
- AUC around 0.5 would mean something's badly wrong (e.g. leakage-free features carry zero signal, or a join is broken)


In [15]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pdf = joined.toPandas()
y = pdf["label"].values
drop_cols = ["loan_id", "Customer_ID", "loan_start_date", "snapshot_date", "label"]
X = pdf.drop(columns=[c for c in drop_cols if c in pdf.columns])

cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

num_imputer = SimpleImputer(strategy="median")
X_train_num = num_imputer.fit_transform(X_train[num_cols])
X_test_num = num_imputer.transform(X_test[num_cols])

cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")
X_train_cat_raw = cat_imputer.fit_transform(X_train[cat_cols])
X_test_cat_raw = cat_imputer.transform(X_test[cat_cols])
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = enc.fit_transform(X_train_cat_raw)
X_test_cat = enc.transform(X_test_cat_raw)

X_train_all = np.hstack([X_train_num, X_train_cat])
X_test_all = np.hstack([X_test_num, X_test_cat])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_all, y_train)
auc = roc_auc_score(y_test, clf.predict_proba(X_test_all)[:, 1])
print(f"sanity-check test AUC: {auc:.3f}")


sanity-check test AUC: 0.500


## Notes / assumptions

- label: bad = 1 if `overdue_amt` > 0 at installment_num == 6 (mob=6 fixed checkpoint), else 0
- point-in-time definition, not cumulative - a loan cured after mob=6 still counts as bad, and vice versa
- feature store built from attributes + financials joined at loan origination date (no leakage) + clickstream left-joined on the same date (~72% coverage, rest kept as null rather than dropped)
- full reasoning is in the docstrings in `utils/data_processing_gold.py` and in the slide deck
